In [2]:
# -*- coding: utf-8 -*-
"""
[1단계] 무속 유튜브 채널 발굴 스크립트
================================================================
검색 API(search.list)로 무속 관련 채널 후보를 발굴하여
'사람이 육안 검수할 수 있는' 후보 목록 CSV를 만듭니다.

- 트랙 A: 무속 × 투자 조합 검색 (투자 콘텐츠 채널 발굴)
- 트랙 B: 순수 무속 키워드 검색 (대조군 발굴, 투자 키워드 비의존 표집)
- 산출물: channel_candidates.csv
    -> 'include' 컬럼에 O 를 표시한 채널만 2단계에서 수집됩니다.

[검수 방법]
1. channel_candidates.csv 를 엑셀로 열기
2. 채널명/설명/샘플 영상 제목을 보고 "무속인 본인 채널"만 include 열에 O 표시
   (뉴스 클립, 예능 짜집기, 일반인 실험 콘텐츠 등은 비워두기)
3. 저장 후 2단계 스크립트(step2) 실행

Quota: search.list = 100유닛/회. 기본 설정 약 56회 ≈ 5,600유닛.
"""

import os
from datetime import datetime, timezone

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# =========================================================
# 0. 사용자 설정
# =========================================================
API_KEY = "YOUR API KEY"           # <-- YouTube Data API v3 키 입력

MAX_SEARCH_CALLS = 60              # 검색 예산 상한 (60회 = 6,000유닛)
MAX_RESULTS_PER_PAGE = 50
SEARCH_ORDER = "viewCount"         # 상위 채널 위주 발굴

OUTPUT_CANDIDATES_CSV = "channel_candidates.csv"

# ---- 트랙 A: 무속 × 투자 조합 (처치군 후보 발굴) ----
TRACK_A_QUERIES = [
    "무당 주식", "신점 주식", "무속인 주식",
    "무당 코인", "신점 코인", "보살 비트코인",
    "점괘 종목", "역술 주식", "무당 금값", "무속인 투자",
]
# 트랙 A 검색 시기 구간 (시기별로 다른 채널이 노출되도록 분산)
TRACK_A_WINDOWS = [
    ("2020-01-01T00:00:00Z", "2021-12-31T23:59:59Z"),
    ("2022-01-01T00:00:00Z", "2023-12-31T23:59:59Z"),
    ("2024-01-01T00:00:00Z", "2025-06-30T23:59:59Z"),
    ("2025-07-01T00:00:00Z", "2026-07-07T23:59:59Z"),
]

# ---- 트랙 B: 순수 무속 키워드 (대조군 후보 발굴) ----
TRACK_B_QUERIES = [
    "신점", "무당 브이로그", "점집 후기", "무속인 일상",
    "굿 의식", "신내림", "무당 일상", "사주 신점",
]
TRACK_B_WINDOWS = [
    ("2020-01-01T00:00:00Z", "2022-12-31T23:59:59Z"),
    ("2023-01-01T00:00:00Z", "2026-07-07T23:59:59Z"),
]

# 채널 설명에서 무속업 힌트를 자동 표시(검수 보조용, 자동 제외는 하지 않음)
SHAMAN_HINT_KEYWORDS = [
    "무당", "무속", "신점", "점사", "보살", "법사", "굿", "신내림",
    "점집", "사주", "역술", "철학관", "타로", "예약",
]


# =========================================================
# 1. 유틸
# =========================================================
def is_quota_error(err: HttpError) -> bool:
    try:
        reason = str(err.content)
        return err.resp.status == 403 and (
            "quotaExceeded" in reason or "dailyLimitExceeded" in reason
        )
    except Exception:
        return False


# =========================================================
# 2. 발굴 검색
# =========================================================
def discover_channels(youtube):
    """
    트랙 A/B 검색을 수행하여 채널 후보 딕셔너리 구성.
    반환: {channel_id: {tracks:set, queries:set, sample_titles:list}}
    """
    candidates = {}
    search_calls = 0
    quota_exceeded = False

    plan = (
        [("A", q, w) for q in TRACK_A_QUERIES for w in TRACK_A_WINDOWS]
        + [("B", q, w) for q in TRACK_B_QUERIES for w in TRACK_B_WINDOWS]
    )
    print(f"[계획] 총 검색 {len(plan)}회 (상한 {MAX_SEARCH_CALLS}회, "
          f"예상 {min(len(plan), MAX_SEARCH_CALLS) * 100:,}유닛)")

    for track, query, (after, before) in plan:
        if search_calls >= MAX_SEARCH_CALLS:
            print(f"[중지] 검색 예산 상한({MAX_SEARCH_CALLS}회) 도달.")
            break
        try:
            resp = youtube.search().list(
                part="snippet", q=query, type="video",
                order=SEARCH_ORDER, maxResults=MAX_RESULTS_PER_PAGE,
                publishedAfter=after, publishedBefore=before,
                regionCode="KR", relevanceLanguage="ko",
            ).execute()
            search_calls += 1
        except HttpError as e:
            if is_quota_error(e):
                print("\n[경고] Quota 초과. 현재까지의 후보로 진행합니다.")
                quota_exceeded = True
                break
            print(f"  [오류] 검색 건너뜀 ({track}/{query}): {e}")
            continue

        new_ch = 0
        for it in resp.get("items", []):
            sn = it.get("snippet", {})
            cid = sn.get("channelId")
            if not cid:
                continue
            entry = candidates.setdefault(
                cid, {"tracks": set(), "queries": set(), "sample_titles": []}
            )
            if cid not in candidates or not entry["tracks"]:
                new_ch += 1
            entry["tracks"].add(track)
            entry["queries"].add(query)
            if len(entry["sample_titles"]) < 3:
                entry["sample_titles"].append(sn.get("title", ""))

        print(f"  [{track}] '{query}' {after[:10]}~{before[:10]}"
              f" -> 채널 누적 {len(candidates)}개 (검색 {search_calls}회)")

    return candidates, search_calls, quota_exceeded


# =========================================================
# 3. 채널 상세 조회 (channels.list, 1유닛/50채널)
# =========================================================
def fetch_channel_details(youtube, candidates: dict) -> list:
    rows = []
    ids = list(candidates.keys())

    for i in range(0, len(ids), 50):
        batch = ids[i:i + 50]
        try:
            resp = youtube.channels().list(
                part="snippet,statistics,contentDetails",
                id=",".join(batch), maxResults=50,
            ).execute()
        except HttpError as e:
            if is_quota_error(e):
                print("[경고] Quota 초과. 상세 미조회 채널은 기본값으로 저장합니다.")
                break
            print(f"  [오류] 채널 배치 건너뜀: {e}")
            continue

        for it in resp.get("items", []):
            cid = it["id"]
            sn = it.get("snippet", {})
            st = it.get("statistics", {})
            uploads = (
                it.get("contentDetails", {})
                .get("relatedPlaylists", {})
                .get("uploads", "")
            )
            desc = sn.get("description", "") or ""
            meta = candidates.get(cid, {})
            tracks = meta.get("tracks", set())

            rows.append({
                "include": "",                                     # <-- 검수 시 O 표시
                "channel_id": cid,
                "channel_title": sn.get("title", ""),
                "discovery_track": "AB" if tracks == {"A", "B"} else "".join(sorted(tracks)),
                "subscriber_count": int(st.get("subscriberCount", 0)) if not st.get("hiddenSubscriberCount") else None,
                "video_count": int(st.get("videoCount", 0)),
                "total_view_count": int(st.get("viewCount", 0)),
                "channel_published_at": sn.get("publishedAt", ""),
                "country": sn.get("country", ""),
                "custom_url": sn.get("customUrl", ""),
                "uploads_playlist_id": uploads,
                "desc_has_shaman_hint": any(k in desc or k in sn.get("title", "") for k in SHAMAN_HINT_KEYWORDS),
                "matched_queries": " | ".join(sorted(meta.get("queries", set()))),
                "sample_video_titles": " | ".join(meta.get("sample_titles", [])),
                "channel_description": desc.replace("\n", " ")[:500],
                "channel_url": f"https://www.youtube.com/channel/{cid}",
            })
    return rows


# =========================================================
# 4. 메인
# =========================================================
def main():
    youtube = build("youtube", "v3", developerKey=API_KEY)

    candidates, search_calls, quota_exceeded = discover_channels(youtube)
    print(f"\n[발굴 완료] 검색 {search_calls}회, 채널 후보 {len(candidates)}개")

    rows = fetch_channel_details(youtube, candidates)
    df = pd.DataFrame(rows)
    if not df.empty:
        # 검수 편의: 무속 힌트 있는 채널 + 구독자 많은 순으로 정렬
        df = df.sort_values(
            ["desc_has_shaman_hint", "subscriber_count"],
            ascending=[False, False], na_position="last",
        ).reset_index(drop=True)

    df.to_csv(OUTPUT_CANDIDATES_CSV, index=False, encoding="utf-8-sig")
    print(f"[저장 완료] {OUTPUT_CANDIDATES_CSV} ({len(df)}개 채널)")
    if not df.empty:
        print(df.groupby("discovery_track")["channel_id"].count().to_string())

    print(
        "\n[다음 할 일]\n"
        f"1. {OUTPUT_CANDIDATES_CSV} 를 엑셀로 열어 무속인 본인 채널만 include 열에 O 표시\n"
        "   (channel_url 클릭해서 확인 가능, 30~50개 패널 권장)\n"
        "2. 저장 후 step2_collect_and_analyze.py 실행"
    )
    if quota_exceeded:
        print("※ Quota 초과로 일부 검색이 생략되었습니다. 필요 시 내일 재실행하여 후보를 보강하세요.")


if __name__ == "__main__":
    main()


[계획] 총 검색 56회 (상한 60회, 예상 5,600유닛)
  [A] '무당 주식' 2020-01-01~2021-12-31 -> 채널 누적 38개 (검색 1회)
  [A] '무당 주식' 2022-01-01~2023-12-31 -> 채널 누적 64개 (검색 2회)
  [A] '무당 주식' 2024-01-01~2025-06-30 -> 채널 누적 88개 (검색 3회)
  [A] '무당 주식' 2025-07-01~2026-07-07 -> 채널 누적 117개 (검색 4회)
  [A] '신점 주식' 2020-01-01~2021-12-31 -> 채널 누적 122개 (검색 5회)
  [A] '신점 주식' 2022-01-01~2023-12-31 -> 채널 누적 134개 (검색 6회)
  [A] '신점 주식' 2024-01-01~2025-06-30 -> 채널 누적 153개 (검색 7회)
  [A] '신점 주식' 2025-07-01~2026-07-07 -> 채널 누적 169개 (검색 8회)
  [A] '무속인 주식' 2020-01-01~2021-12-31 -> 채널 누적 175개 (검색 9회)
  [A] '무속인 주식' 2022-01-01~2023-12-31 -> 채널 누적 184개 (검색 10회)
  [A] '무속인 주식' 2024-01-01~2025-06-30 -> 채널 누적 194개 (검색 11회)
  [A] '무속인 주식' 2025-07-01~2026-07-07 -> 채널 누적 204개 (검색 12회)
  [A] '무당 코인' 2020-01-01~2021-12-31 -> 채널 누적 212개 (검색 13회)
  [A] '무당 코인' 2022-01-01~2023-12-31 -> 채널 누적 221개 (검색 14회)
  [A] '무당 코인' 2024-01-01~2025-06-30 -> 채널 누적 230개 (검색 15회)
  [A] '무당 코인' 2025-07-01~2026-07-07 -> 채널 누적 242개 (검색 16회)
  [A] '신점 코인' 2020-01-01~2021